# Phase 0.7 round 2 — continue SmolVLA on D1

**This continues round 1's SmolVLA on the new data.** It does not train from scratch and it does not
start from `smolvla_base`: the starting weights are `HALDijkstraaa/smolvla_tray_cylinder_s1000_trim`,
the policy that scored 6/18 in [round 1](Details/phase07_portable_rig.md).

| | |
|---|---|
| Dataset | `so101_tray_cylinder_varied_20261013_082911_trimmed` — 50 episodes, **head-trimmed, 2 s tail kept**, 33,337 frames |
| Starts from | `smolvla_tray_cylinder_s1000_trim` (round 1's final, 20k steps on D0) |
| Steps | **20,000** at batch 64 = 1,280,000 samples = **42.5 epochs of D1** |
| Learning rate | **1e-5** — lowered 10× from the preset, see below |
| Checkpoints | every 5,000 → 4, each pushed to the Hub as it is written |
| GPU | **L4 or better.** Round 1 peaked at **16.5 of 24 GiB** at batch 64 — a T4 is too small |
| Expected | **~14 h on an L4**, ~5 h on an A100 — Part B measures it in ~10 min and prints the real figure |
| Output | `HALDijkstraaa/smolvla_tray_cylinder_s1000_d1` (private) |

### The learning rate is the one thing that would silently ruin this

SmolVLA's preset is a cosine from peak **1e-4** down to 2.5e-6 over `scheduler_decay_steps`, after
1,000 warmup steps — and **a new run restarts that schedule from the top.** Round 1 *finished* at
2.5e-6, so accepting the default would hand the converged policy 1e-4 again: **40× its final rate**,
which would largely undo the warm start. The config cell therefore passes
`--policy.optimizer_lr=1e-5`, dropping the peak 10×. The cosine still runs, pinned to the real step
count, so the run anneals rather than sitting flat.

### The 5,000-step checkpoint is the one to compare against ACT

Steps are the wrong unit across policies with different batch sizes. At batch 64, **5,000 steps is
320,000 samples — exactly what ACT's whole 40,000-step run at batch 8 sees.** So the 5k checkpoint is
sample-matched to ACT and is where the clean comparison is read; 10k/15k/20k are the train-longer arm.

### What this costs the pre-registration

[§12.1](Details/phase07_portable_rig.md) pre-registered P1–P3 against *fresh* training on D1.
Continuing instead means this policy has seen D0 **and** D1, so a change in the ACT−SmolVLA gap can no
longer be attributed to D1's character alone — P1 and P2 are weakened, and **P3 is not measurable at
all**. The trade is deliberate: this is the faster route to a usable policy.

---

## 1 — Runtime check


In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", GPU)
assert GPU, "No GPU visible. On Colab: Runtime > Change runtime type > GPU (L4 or A100)."
if "T4" in GPU:
    print("\n!! T4 (16 GB): TOO SMALL. SmolVLA peaked at 16.5 GiB at batch 64 in round 1.")
    print("   Part B will OOM and step the batch down, which changes the recipe and breaks the")
    print("   sample-matched comparison with ACT. Switch to an L4.")
elif "L4" in GPU:
    print("\nL4 (24 GB): fits - round 1 ran this exact recipe here at 16.5 GiB, no step-down.")
    print("            ~14 h for 20,000 steps. Speed is the constraint, not memory.")
elif "A100" in GPU:
    print("\nA100: fits easily and is ~2.5-3x faster - roughly 5 h instead of 14.")
    print("      Note Colab bills A100 at several times the L4 rate, so the wall-clock saving")
    print("      is real but the compute-unit saving may not be.")


## 2 — Install

Colab already provides a working CUDA torch, so only lerobot is installed — pinned to **v0.6.1**, the
version that recorded the dataset. (The 4090 notebook installs torch itself because that host is bare;
here, letting Colab's torch stand is both simpler and less likely to break.)

In [ ]:
!pip install -q "lerobot[smolvla,dataset,training]==0.6.1"
import lerobot, torch
print("lerobot", lerobot.__version__, "| torch", torch.__version__, "| cuda", torch.version.cuda)


> If `import lerobot` fails, restart the runtime (**Runtime > Restart session**) and re-run this cell.
> Nothing later imports lerobot into this kernel — every use goes through a subprocess.

## 3 — Authentication

**A Colab secret alone is not enough.** `huggingface_hub` resolves a token as OIDC → `HF_TOKEN` env var →
token file → Colab secret, and that last path calls `google.colab.userdata.get()`, which needs the
kernel's channel to the Colab frontend. `lerobot-train` is a **subprocess**, where that channel does not
exist — so with only a secret set, `whoami()` succeeds in the notebook while training authenticates as
nobody. These cells copy the token into the environment *and* the token file, then prove a subprocess can
use it.

The run also creates a new model repo. A fine-grained token can be granted `repo.write` either globally
or **scoped to your own user** — the latter is the least-privilege setup and shows up nowhere under
`global`, so the cell checks both and only *warns* if it finds neither. Part B settles it for real by
pushing a checkpoint and verifying it landed.

In [ ]:
import os, sys, subprocess
from huggingface_hub import get_token, login, whoami

tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
    except Exception:
        tok = None
if not tok:
    from huggingface_hub import notebook_login
    notebook_login(); tok = get_token()
assert tok, "no HF token - add a Colab secret named HF_TOKEN, or run notebook_login()"

login(token=tok, add_to_git_credential=False)     # writes ~/.cache/huggingface/token
os.environ["HF_TOKEN"] = tok                      # set AFTER login, so it does not warn that the
                                                  # env var overrides what was just configured
                                                  # (same token either way); subprocesses inherit it

r = subprocess.run([sys.executable, "-c",
                    "from huggingface_hub import whoami; print(whoami()['name'])"],
                   capture_output=True, text=True)
assert r.returncode == 0, "a subprocess still cannot authenticate:\n" + r.stderr[-800:]
me = whoami()
print("HF ok - notebook:", me["name"], "| subprocess:", r.stdout.strip())


def hf_write_grant(me):
    """Report how (or whether) this token is allowed to create repos in the user's namespace.

    A fine-grained token carries permissions either globally or SCOPED TO AN ENTITY. The
    least-privilege setup - and the one the Hub's UI steers you to - is `repo.write` scoped to your
    own user, which does not appear under `global` at all. Checking only `global` therefore rejects
    a perfectly good token, which is exactly what an earlier version of this cell did.
    """
    auth = me.get("auth", {}).get("accessToken", {})
    if auth.get("role") != "fineGrained":
        return True, f"{auth.get('role', 'write')} token"
    fg = auth.get("fineGrained") or {}
    if "repo.write" in (fg.get("global") or []):
        return True, "global repo.write"
    for s in fg.get("scoped") or []:
        ent = s.get("entity") or {}
        if (ent.get("type") in ("user", "org") and ent.get("name") == me["name"]
                and "repo.write" in (s.get("permissions") or [])):
            return True, f"repo.write scoped to {ent['type']} '{ent['name']}'"
    return False, f"no repo.write found - global={fg.get('global')}"


ok, how = hf_write_grant(me)
print("token write access:", how)
if not ok:
    # Deliberately a warning, not an assertion. The Hub's permission model has more shapes than it
    # is wise to enumerate here, and Part B proves the real thing 15 minutes from now by pushing a
    # checkpoint and checking it landed. A hard gate here can only produce false blocks.
    print("\n!! Could not confirm this token can create the model repo this run pushes to.")
    print("   If Part B's 'checkpoint uploaded to the Hub' check fails, that is the reason - give")
    print("   the token repo.write (globally, or scoped to your user) at")
    print("   https://huggingface.co/settings/tokens")

In [ ]:
import os, sys, subprocess, wandb

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key; wandb.login(key=key)
else:
    wandb.login()                                  # key from https://wandb.ai/authorize
    os.environ["WANDB_API_KEY"] = wandb.api.api_key

r = subprocess.run([sys.executable, "-c", "import wandb; print(bool(wandb.api.api_key))"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "True", "a subprocess cannot see the wandb key:\n" + r.stderr[-500:]
print("wandb ok in notebook and subprocess")

## 4 — Configuration

Everything the run needs, in one cell. The only differences from round 1 are the dataset, the starting weights and the learning rate.


In [ ]:
import collections, json, os, re, shutil, subprocess, sys, threading, time
from pathlib import Path

# Colab has one big disk and no separate data volume, so no redirection is needed.
WORK = Path("/content")
OUTPUT_ROOT = WORK / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_tray_cylinder_varied_20261013_082911_trimmed"
PROJECT = "phase07-policy-transfer"
SEED    = 1000

# Held out for validation loss. ROUND 2 NOTE: D1 randomises the cylinder position continuously, so
# the 9-block structure that forced a hand-picked holdout in round 1 is gone - the verifier measures
# 1x block separation where D0 had 27x, and lerobot's default "last N" split would now be perfectly
# fair. The same indices are kept anyway so the split MECHANISM is identical to round 1 and the only
# thing differing between the two runs is the data. They are arbitrary here, which is the point.
HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

EVAL_SPLIT, LOG_FREQ = 0.1, 200

# A SmolVLA checkpoint is ~1.3 GB, so four of them plus the base model is a few GB on a Colab disk
# with ~68 GB free. Nothing needs reaping; keep them all.
KEEP_ALL_CHECKPOINTS = True
KEEP_CKPTS = 2

# ---- SmolVLA round 2 ----------------------------------------------------------------------------
JOB   = "smolvla_tray_cylinder_s1000_d1"
# ROUND 2 IS A CONTINUATION. BASE_MODEL is round 1's finished policy, NOT lerobot/smolvla_base.
BASE_MODEL = f"{HF_USER}/smolvla_tray_cylinder_s1000_trim"
# Round 1's recipe, unchanged: batch 64. STEPS is 20,000 further steps over D1.
BASE_BATCH, BASE_STEPS = 64, 20000
BATCH, STEPS = BASE_BATCH, BASE_STEPS
SAVE_FREQ = 5000      # 4 checkpoints; the 5,000 one is sample-matched to ACT's whole run
EVAL_STEPS = 5000     # held-out loss cadence; training is unaffected either way.
# SAVE_FREQ must be a multiple of EVAL_STEPS, so that every checkpoint has an eval at its own
# step: lerobot tests `step % eval_steps == 0` and `step % save_freq == 0` independently, and
# runs the eval first, so a mismatched pair leaves some checkpoints with no loss beside them.
assert SAVE_FREQ % EVAL_STEPS == 0, f"{SAVE_FREQ} is not a multiple of {EVAL_STEPS}"

# A WARM START NEEDS A LOWER PEAK LR. SmolVLA's preset cosine runs from 1e-4 down to 2.5e-6, and a
# new run restarts it from the top - 40x the rate round 1 ended on, which would undo the warm start.
OPTIM_LR = "1e-5"

# The checkpoint declares its own camera names (inherited from smolvla_base); the batch is renamed
# to match. Without this the run dies on a feature mismatch.
RENAME = {
    "observation.images.top": "observation.images.camera1",
    "observation.images.wrist": "observation.images.camera2"
}

NUM_WORKERS = min(8, os.cpu_count() or 2)


def train_cmd(job=None, steps=None, batch=None, extra=()):
    """Build the lerobot-train argv.

    Every default is resolved from the globals at CALL time, not bound at definition time, so a
    batch step-down in Part B reaches a later bare train_cmd() call. Checkpoints go to the Hub every
    SAVE_FREQ steps, so a disconnect costs at most that many.
    """
    job = JOB if job is None else job
    steps = STEPS if steps is None else steps
    batch = BATCH if batch is None else batch
    return ["lerobot-train",
        f"--dataset.repo_id={DATASET}", f"--dataset.episodes={EPISODES}",
        f"--dataset.eval_split={EVAL_SPLIT}", f"--eval_steps={EVAL_STEPS}",
        *( [f"--dataset.video_backend={globals()['VIDEO_BACKEND']}"]
           if globals().get("VIDEO_BACKEND") else [] ),
        # --policy.path is what makes this a CONTINUATION: it loads weights into a new run with a
        # fresh optimizer and step counter. (--resume=true is a different thing - it restores
        # optimizer state to continue an interrupted run, which is not what is wanted here.)
        f"--policy.path={BASE_MODEL}",
        f"--rename_map={json.dumps(RENAME)}",

        "--policy.device=cuda",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        # Pin the cosine to the real step count so the schedule completes, and lower its peak for
        # the warm start (see OPTIM_LR above).
        f"--policy.scheduler_decay_steps={steps}",
        f"--policy.optimizer_lr={OPTIM_LR}",
        f"--batch_size={batch}", f"--steps={steps}", f"--num_workers={NUM_WORKERS}",
        f"--seed={SEED}", f"--save_freq={SAVE_FREQ}", f"--log_freq={LOG_FREQ}",
        f"--output_dir={OUTPUT_ROOT}/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]


KEEP = ("step:", "eval_loss", "Train/eval split", "num_learnable_params", "input_features",
        "Traceback", "Error", "error:", "Track this run", "End of training", "out of memory")


def gpu_watch(stop, peak):
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip().splitlines()
        if out:
            peak[0] = max(peak[0], int(out[0]))
        time.sleep(5)


def disk_free_gb(path=None):
    # Falls through so a bad path can never raise inside the watchdog loop, where an exception
    # would silence the very warning this exists to produce.
    for p in ([str(path)] if path else []) + [str(WORK), "/", "."]:
        try:
            s = shutil.disk_usage(p)
            return s.free / 1e9, s.used / 1e9
        except Exception:
            continue
    return float("nan"), float("nan")


def dir_size_gb(path):
    try:
        return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e9
    except Exception:
        return 0.0


def watch_disk(stop, out_dir, total_steps, save_freq, period=120):
    """Report disk as the run proceeds and shout BEFORE it runs out.

    Every checkpoint is kept (KEEP_ALL_CHECKPOINTS), so the footprint grows all run: for pi0.5 that
    is ~11.5 GB every save_freq steps. Rather than discover the limit at hour 14, this measures the
    actual size of the checkpoints written so far, extrapolates to the number still to come, and
    compares that against free space.
    """
    d = Path(out_dir) / "checkpoints"
    warned = False
    while not stop.is_set():
        try:
            free, used = disk_free_gb(out_dir)
            done = sorted(int(p.name) for p in d.iterdir()
                          if d.is_dir() and p.is_dir() and not p.is_symlink() and p.name.isdigit()) \
                   if d.is_dir() else []
            if done:
                each = dir_size_gb(d) / len(done)
                remaining = max(0, (total_steps // save_freq) - len(done))
                need = each * remaining
                msg = (f"    [disk] {len(done)} checkpoint(s) written, {each:.1f} GB each | "
                       f"{remaining} to go needs ~{need:.0f} GB | {free:.0f} GB free")
                if need > free * 0.9 and not warned:
                    warned = True
                    print("\n" + "!" * 78, flush=True)
                    print(f"  DISK WARNING: the remaining {remaining} checkpoint(s) need "
                          f"~{need:.0f} GB but only {free:.0f} GB is free.", flush=True)
                    print(f"  This run will stop partway unless you act. Options, cheapest first:", flush=True)
                    print(f"    - grow the data disk (this host charges per GB-day);", flush=True)
                    print(f"    - delete older local checkpoints by hand - they are already on the", flush=True)
                    print(f"      Hub, so nothing is lost:", flush=True)
                    print(f"        import shutil; shutil.rmtree('{d}/<step>')", flush=True)
                    print(f"    - or set KEEP_ALL_CHECKPOINTS = False and restart the cell.", flush=True)
                    print("!" * 78 + "\n", flush=True)
                else:
                    print(msg, flush=True)
        except Exception as e:
            print(f"    [disk] watchdog pass failed ({type(e).__name__}); will retry", flush=True)
        stop.wait(period)


def reap_checkpoints(stop, out_dir, keep=KEEP_CKPTS):
    """Delete superseded LOCAL checkpoints. Only runs when KEEP_ALL_CHECKPOINTS is False.

    Each checkpoint is the full model plus optimizer state - for pi0.5 ~11.5 GB - and lerobot never
    prunes them. They are redundant locally the moment save_checkpoint_to_hub has pushed them, which
    is what makes deleting them safe. Keeps the newest `keep` numbered directories and never touches
    `last`, the symlink resume follows.
    """
    d = Path(out_dir) / "checkpoints"
    while not stop.is_set():
        try:
            if d.is_dir():
                steps = sorted((p for p in d.iterdir() if p.is_dir() and not p.is_symlink()
                                and p.name.isdigit()), key=lambda p: int(p.name))
                dropped = []
                for p in (steps[:-keep] if len(steps) > keep else []):
                    shutil.rmtree(p, ignore_errors=True)   # per-item, so one failure is not fatal
                    dropped.append(p.name)
                if dropped:
                    print(f"    [disk] removed local checkpoint(s) {', '.join(dropped)} "
                          f"(already on the Hub); {disk_free_gb()[0]:.0f} GB free", flush=True)
        except Exception as e:
            print(f"    [disk] reaper pass failed ({type(e).__name__}); will retry", flush=True)
        stop.wait(60)


def run(cmd, quiet=True):
    """Stream a run, dropping tqdm spam, and record peak GPU memory alongside throughput.

    Markers are extracted AS EACH LINE STREAMS BY, not by searching a buffer afterwards. A real run
    emits far more output than it is safe to hold in memory, and everything the smoke test checks -
    the train/eval split, the wandb URL, the parameter count - is printed in the first few seconds,
    so a bounded tail buffer would have evicted all of it long before the run ended.
    """
    t0 = time.time()
    head, tail = [], collections.deque(maxlen=120)   # head for context, tail for failure diagnosis
    peak, stop = [0], threading.Event()
    seen = {"split": None, "wandb": None, "params": None}
    rates, losses, eval_losses = [], [], []
    threading.Thread(target=gpu_watch, args=(stop, peak), daemon=True).start()
    out_dir = next((c.split("=", 1)[1] for c in cmd if c.startswith("--output_dir=")), None)
    n_steps = next((int(c.split("=", 1)[1]) for c in cmd if c.startswith("--steps=")), STEPS)
    if out_dir:
        if KEEP_ALL_CHECKPOINTS:
            threading.Thread(target=watch_disk, args=(stop, out_dir, n_steps, SAVE_FREQ),
                             daemon=True).start()
        else:
            threading.Thread(target=reap_checkpoints, args=(stop, out_dir), daemon=True).start()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    oom = False
    for line in p.stdout:
        if len(head) < 400:
            head.append(line)
        tail.append(line)
        if "out of memory" in line.lower():
            oom = True
        m = re.search(r"Train/eval split: ([0-9]+) train, ([0-9]+) eval", line)
        if m:
            seen["split"] = (int(m.group(1)), int(m.group(2)))
        m = re.search(r"(https://wandb\.ai/[^\s]+)", line)
        if m and not seen["wandb"]:
            seen["wandb"] = m.group(1)
        m = re.search(r"num_learnable_params=([0-9]+)", line)
        if m:
            seen["params"] = int(m.group(1))
        # tqdm FLIPS THE UNIT once an iteration takes over a second: ACT logs "4.47step/s" but
        # pi0.5 logs "1.95s/step". Parse both and normalise to steps/s, or the slow policies
        # report no throughput at all and the readiness verdict fails on a healthy run.
        rates += [float(x) for x in re.findall(r"([0-9.]+) ?step/s", line)]
        rates += [1.0 / float(x) for x in re.findall(r"([0-9.]+) ?s/step", line) if float(x) > 0]
        # " loss:" with the leading space on purpose: it must not also match l1_loss: or kld_loss:
        losses += [float(x) for x in re.findall(r" loss:([0-9.]+)", line)]
        eval_losses += [float(x) for x in re.findall(r"eval_loss=([0-9.]+)", line)]
        if (not quiet) or any(k in line for k in KEEP) or "wandb.ai" in line:
            sys.stdout.write(line); sys.stdout.flush()
    p.wait(); stop.set()
    dt = (time.time() - t0) / 3600
    free, used = disk_free_gb()
    print(f"[exit {p.returncode} after {dt:.2f} h | peak GPU {peak[0]/1024:.1f} GiB "
          f"| disk {used:.0f} GB used, {free:.0f} GB free]")
    if p.returncode != 0:
        print("--- last lines ---"); print("".join(tail)[-2500:])
    return {"code": p.returncode, "hours": dt, "peak_gib": peak[0] / 1024, "oom": oom,
            "rates": rates, "losses": losses, "eval_losses": eval_losses, **seen,
            "head": "".join(head), "tail": "".join(tail)}


print("holdout      :", HOLDOUT)
print("episode order:", EPISODES[:6], "...", EPISODES[-6:])
print("workers      :", NUM_WORKERS)

## 5 — Preflight

Three things that each cost minutes now and hours later: the dataset exists and carries its
codebase-version git tag; the holdout reorder is well formed; and the camera keys line up with what the
policy expects. None of this needs a GPU.

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
checks = []

# SmolVLA's backbone is NOT gated, so the pi0.5 PaliGemma access check is gone from here.
# What remains is the part that matters for any pretrained policy: that the dataset is
# really there and tagged, and that the renamed camera keys line up with the checkpoint's.

ok_ds = api.repo_exists(DATASET, repo_type="dataset")
checks.append(("trimmed dataset exists", ok_ds))
assert ok_ds, f"{DATASET} not found. Run colab_phase07_prep_trim.ipynb (Stage 2) first."

info = json.loads(open(hf_hub_download(DATASET, "meta/info.json", repo_type="dataset")).read())
tags = [t.name for t in api.list_repo_refs(DATASET, repo_type="dataset").tags]
# lerobot resolves a dataset at the revision named by its codebase_version; without the tag, training
# dies several minutes in with "Your dataset must be tagged with a codebase version".
checks.append((f"codebase-version tag {info['codebase_version']}", info["codebase_version"] in tags))
checks.append(("50 episodes", info["total_episodes"] == 50))
checks.append(("idle-trimmed (fewer frames than D1's raw 38,389)", info["total_frames"] < 38389))
print(f"dataset: {info['total_episodes']} episodes, {info['total_frames']} frames, "
      f"{info['fps']} fps, tags={tags}")

checks.append(("holdout reorder is a permutation of 0-49", sorted(EPISODES) == list(range(50))))
checks.append(("holdout is the tail of the episode list", EPISODES[-len(HOLDOUT):] == HOLDOUT))

ds_cams = {k for k in info["features"] if k.startswith("observation.images.")}
# The decisive check for a pretrained policy: after renaming, the batch's camera keys must be a subset
# of what the checkpoint declares. lerobot accepts a subset in either direction, and anything the policy
# declares but the batch does not supply is padded internally.
pol_cfg = json.loads(open(hf_hub_download(BASE_MODEL, "config.json")).read())
pol_cams = {k for k, v in pol_cfg["input_features"].items() if v["type"] == "VISUAL"}
renamed = {RENAME.get(k, k) for k in ds_cams}
print("\npolicy declares :", sorted(pol_cams))
print("dataset supplies:", sorted(ds_cams))
print("after rename    :", sorted(renamed))
print("left unfed      :", sorted(pol_cams - renamed), "(padded internally with a zero mask)")
checks += [
    ("rename map keys exist in the dataset", set(RENAME) <= ds_cams),
    ("renamed keys are a subset of the policy's", renamed <= pol_cams),
]
for k, v in sorted(pol_cfg.items()):
    if k in ("freeze_vision_encoder", "train_expert_only", "optimizer_lr", "chunk_size",
             "n_action_steps", "num_steps", "resize_imgs_with_padding", "empty_cameras"):
        print(f"  base config  {k:28s} = {v}")

print()
for n, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n}")
assert all(ok for _, ok in checks), "preflight failed - fix before running Part B"
print("\npreflight OK")

---
# Part B — Smoke test

One cell, ~15 minutes. Runs **300 real training steps** — the balanced split, periodic validation, wandb,
and a checkpoint pushed to the Hub — then checks every result and prints a readiness verdict plus the
projected runtime for the full run on *your* GPU.

If it hits CUDA OOM it **halves the batch and retries**, then scales the step count to hold
samples-seen fixed, so the published recipe's sample budget survives a smaller GPU. Whatever it settles
on is what Part C uses.

It fails loudly rather than warning, so Part C cannot start on a broken setup.

In [ ]:
from huggingface_hub import HfApi

SMOKE_REPO, N = f"{HF_USER}/smolvla_r2_smoke_delete_me", 300
batch, res = BATCH, None

while True:
    print(f"\n{'=' * 70}\n  smoke test at batch {batch}\n{'=' * 70}")
    shutil.rmtree(OUTPUT_ROOT / "smoke", ignore_errors=True)
    cmd = [c for c in train_cmd(job="smoke", steps=N, batch=batch)
           if not c.startswith(("--policy.repo_id", "--save_freq", "--eval_steps"))]
    cmd += [f"--policy.repo_id={SMOKE_REPO}", f"--eval_steps={N // 2}", f"--save_freq={N}"]
    res = run(cmd)

    if res["oom"] and batch > 1:
        print(f"\n!! OOM at batch {batch}. Halving and retrying.")
        print("   Steps will be scaled to keep samples-seen constant, so the recipe's sample budget")
        print("   is preserved even though the batch is smaller.")
        batch //= 2
        continue
    break

# Every check reports what was actually OBSERVED, so a failure is diagnosable rather than a bare
# FAIL. The values come from run(), which extracts them while the output streams past - searching a
# buffer afterwards is what made the first version report false failures, because the lines it
# looked for scroll away in the first seconds of a run.
split = res["split"]
checks = [
    ("training completed",
     res["code"] == 0),
    (f"balanced split 45 train / 5 eval   (saw {split})",
     split == (45, 5)),
    (f"validation ran   ({len(res['eval_losses'])} eval_loss points: {res['eval_losses']})",
     len(res["eval_losses"]) > 0),
    (f"wandb connected   ({res['wandb'] or 'NO run URL seen'})",
     bool(res["wandb"])),
]
try:
    files = HfApi().list_repo_files(SMOKE_REPO, repo_type="model")
    checks.append(("checkpoint uploaded to the Hub", any("checkpoints/" in f for f in files)))
except Exception as e:
    checks.append((f"checkpoint uploaded to the Hub ({type(e).__name__})", False))

losses, ev = res["losses"], res["eval_losses"]
if len(losses) >= 2:
    checks.append((f"train loss fell ({losses[0]:.3f} -> {losses[-1]:.3f})", losses[-1] < losses[0]))
if len(ev) >= 2:
    checks.append((f"eval loss fell ({ev[0]:.4f} -> {ev[-1]:.4f})", ev[-1] < ev[0]))

print()
for n_, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {n_}")

BATCH = batch
# Derived from the published recipe, not from the current STEPS, so re-running this cell after a
# step-down does not scale the step count a second time.
STEPS = int(round(BASE_STEPS * BASE_BATCH / BATCH))
rates = res["rates"]
rate = sorted(rates)[len(rates) // 2] if rates else None

print(f"\n{'=' * 70}")
print(f"  GPU            : {GPU}")
print(f"  peak memory    : {res['peak_gib']:.1f} GiB")
if rate:
    hours = STEPS / rate / 3600
    print(f"  throughput     : {rate:.2f} step/s  ({rate * BATCH:.0f} samples/s)")
    print(f"  batch          : {BATCH}"
          + ("" if BATCH == BASE_BATCH else f"   (stepped down from {BASE_BATCH})"))
    print(f"  steps          : {STEPS:,}   ({STEPS * BATCH / 1000:.0f}k samples)")
    print(f"  PROJECTED RUN  : {hours:.1f} h  (+ validation and checkpoint pushes, roughly 5%)")
    if hours > 12:
        print("\n  !! Over 12 h. Colab will likely disconnect at least once - Part C resumes from")
        print(f"     the last Hub checkpoint, losing at most {SAVE_FREQ:,} steps "
              f"({SAVE_FREQ / rate / 3600:.1f} h), so that is survivable.")
        print("     To shorten it instead, set STEPS by hand (the cosine schedule follows it) or")
        print("     try --policy.gradient_checkpointing=false if peak memory leaves room.")

    # Disk is the other thing that ends a long VLA run. Each checkpoint is the whole model plus
    # optimizer state, and the smoke test has just written exactly one, so measure rather than guess.
    ck = OUTPUT_ROOT / "smoke" / "checkpoints"
    sz = 0
    if ck.is_dir():
        sz = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9
    free, used = disk_free_gb()
    n_ck = max(1, STEPS // SAVE_FREQ)
    print(f"\n  disk now       : {used:.0f} GB used, {free:.0f} GB free")
    if sz:
        # Everything already on disk stays there (KEEP_ALL_CHECKPOINTS), so what matters is whether
        # the checkpoints still to be written fit in what is free RIGHT NOW.
        need = sz * n_ck
        fits = need < free
        print(f"  per checkpoint : {sz:.1f} GB  x {n_ck} checkpoints over the run = {need:.0f} GB")
        print(f"  keeping all    : {'FITS' if fits else 'WILL NOT FIT'}  "
              f"({need:.0f} GB needed vs {free:.0f} GB free, "
              f"{free - need:+.0f} GB margin at the end)")
        if not fits:
            print(f"\n  !! The run would stop partway. Either grow the data disk by "
                  f"~{need - free + 10:.0f} GB, raise SAVE_FREQ to write fewer checkpoints, or set")
            print(f"     KEEP_ALL_CHECKPOINTS = False to retain only the newest {KEEP_CKPTS}.")
        elif free - need < 2 * sz:
            print(f"     Margin is under two checkpoints - the watchdog in Part C will warn you if")
            print(f"     it tightens. Deleting an early checkpoint by hand is always safe; they are")
            print(f"     all on the Hub.")
        print(f"  Hub repo ends  : ~{need:.0f} GB, since every checkpoint is pushed (Part D prunes).")
else:
    print("  throughput     : could not parse step/s from the log")

SMOKE_OK = all(ok for _, ok in checks) and rate is not None
print(f"\n  READY: {SMOKE_OK}")
print("=" * 70)
if SMOKE_OK:
    # Reclaim the smoke run's own checkpoint before Part C starts. For pi0.5 that is ~11.5 GB,
    # a meaningful slice of the margin when every checkpoint is being kept.
    _before = disk_free_gb()[0]
    shutil.rmtree(OUTPUT_ROOT / "smoke", ignore_errors=True)
    _after = disk_free_gb()[0]
    print(f"\nsmoke output cleaned up: {_after - _before:+.1f} GB reclaimed, {_after:.0f} GB free")
    print("\nPaste the block above, then run Part C.")
    print(f"Clean up when convenient:  HfApi().delete_repo('{SMOKE_REPO}', repo_type='model')")
else:
    raise RuntimeError("smoke test failed - see the FAIL lines above. Do not start Part C.")

---
# Part C — Run it

**One cell. Run it and walk away.**

**Restartable.** If Colab disconnects, re-run this same cell. It first reads the run's Hub repo: a final
`model.safetensors` means finished (skip), checkpoints without one mean part-done (resume from the last,
losing at most 5,000 steps), neither means start. So a disconnect never repeats completed
work, and you never have to work out where it got to.

If you reconnected and `SMOKE_OK` is unset, that is expected — re-running Part B costs 15 minutes and
proves nothing new, so this cell warns rather than blocking.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

_api = HfApi()

if not globals().get("SMOKE_OK"):
    print("!! Part B has not run in this kernel (SMOKE_OK is unset).")
    print("   Expected after a reconnect: this cell is restartable and re-checks the Hub itself.")
    print("   On a fresh setup, run Part B first - it must print READY: True.\n")


def run_state(job):
    """Where this run stands, read from its Hub repo: finished, part-done, or not started."""
    try:
        files = _api.list_repo_files(f"{HF_USER}/{job}", repo_type="model")
    except Exception:
        return "new", None
    if "model.safetensors" in files:            # the final push only happens at the very end
        return "done", None
    steps = sorted({f.split("/")[1] for f in files
                    if f.startswith("checkpoints/") and f.split("/")[1].isdigit()})
    return ("resume", steps[-1]) if steps else ("new", None)

t0 = time.time()
where, last = run_state(JOB)

if where == "done":
    print(f"already finished on the Hub - nothing to do: https://huggingface.co/{HF_USER}/{JOB}")
    res = {"code": 0, "hours": 0.0}
elif where == "resume":
    # Prefer the local checkpoint if this VM still has it; otherwise pull from the Hub. The step
    # counter, optimizer state, episode order, holdout and seed all come back from the checkpoint -
    # only --steps is read from the CLI.
    local = OUTPUT_ROOT / JOB / "checkpoints/last/pretrained_model/train_config.json"
    src = str(local) if local.exists() else f"{HF_USER}/{JOB}"
    print(f"resuming from step {last} ({'local' if local.exists() else 'Hub'})")
    cmd = ["lerobot-train", f"--config_path={src}", "--resume=true", f"--steps={STEPS}"]
    if not local.exists():
        cmd.append(f"--output_dir={OUTPUT_ROOT}/{JOB}_resumed")
    res = run(cmd)
else:
    print(f"starting fresh: {STEPS:,} steps at batch {BATCH}")
    res = run(train_cmd(steps=STEPS, batch=BATCH))

status = "ok" if res["code"] == 0 else f"FAILED (exit {res['code']})"
print(f"\n{'=' * 70}")
print(f"  {JOB}")
print(f"  status : {status}")
print(f"  time   : {res['hours']:.2f} h")
print(f"  model  : https://huggingface.co/{HF_USER}/{JOB}")
print(f"  wandb  : project {PROJECT}, run {JOB}")
print("=" * 70)
if res["code"] != 0:
    print("\nRe-run this cell to resume from the last Hub checkpoint, or see Part D.")
else:
    print("\nPaste this summary back. Then run the other two policies' notebooks if they are not")
    print("already going, and record the result in Details/phase07_training.md.")

---

## Done

The run pushes every checkpoint to `HALDijkstraaa/smolvla_tray_cylinder_s1000_d1` as it goes, so a dead Colab session costs at most `SAVE_FREQ` = 5,000 steps and Part C resumes from the Hub.

**Next:** read the eval curve, then roll out. Compare the **5,000-step** checkpoint against ACT's 40,000-step run - those are sample-matched at 320,000 samples - and the 20,000-step one as the train-longer arm. Round 1's lesson is that the loss-optimal checkpoint was *not* the robot-optimal one, so do not pick on loss alone.
